In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import os

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

from joblib import Parallel, delayed

DATA_PATH = "Data/crsp_ff3_daily_1995_2024.parquet"
VAL_DATES_PATH = "Data/val_dates_hyperparam.csv"

features = ["mktrf", "smb", "hml"]
target = "excess_ret"

TEST_START, TEST_END = "2024-01-01", "2024-12-31"

#Params
BEST_PARAMS = {
  "hidden_dim": 32,
  "lr": 0.001,
  "weight_decay": 0.0005,
  "epochs": 200
}

N_JOBS = max(1, (os.cpu_count() or 8) // 2)



Pre-2024 (train+val): (13064388, 11)
Test 2024: (684286, 11)
Val dates loaded (for record): 252
N_JOBS: 6


In [ ]:
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

# Load val dates used for tuning (NOT used to filter training in this standard run)
val_dates = pd.to_datetime(pd.read_csv(VAL_DATES_PATH)["date"])

# Test set = 2024
test = df[(df["date"] >= TEST_START) & (df["date"] <= TEST_END)].copy()

# Pre-2024 pool = all data available to train final model
pre = df[df["date"] <= "2023-12-31"].copy()

print("Pre-2024 (train+val):", pre.shape)
print("Test 2024:", test.shape)
print("Val dates loaded (for record):", len(val_dates))
print("N_JOBS:", N_JOBS)


In [6]:
class ShallowNN(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

def fit_ff3_ols(stock_train: pd.DataFrame):
    X = sm.add_constant(stock_train[features])
    y = stock_train[target]
    return sm.OLS(y, X).fit()

def train_nn_train(stock_train: pd.DataFrame, params: dict):
    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(stock_train[features].values)

    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(stock_train[[target]].values).flatten()

    model = ShallowNN(input_dim=X_train.shape[1], hidden_dim=int(params["hidden_dim"]))
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"])
    )
    loss_fn = nn.MSELoss()

    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    epochs = int(params["epochs"])
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_t)
        loss = loss_fn(preds, y_t)
        loss.backward()
        optimizer.step()

    return model, x_scaler, y_scaler

def process_one_permno(permno: int, train_df: pd.DataFrame, test_df: pd.DataFrame, params: dict):
    stock_train = train_df[train_df["permno"] == permno]
    stock_test  = test_df[test_df["permno"] == permno]

    if stock_test.empty or len(stock_train) < 300:
        return None

    # FF3
    ff3_model = fit_ff3_ols(stock_train)
    X_test_ff3 = sm.add_constant(stock_test[features])
    ff3_pred = ff3_model.predict(X_test_ff3).values

    # NN
    nn_model, x_scaler, y_scaler = train_nn_train(stock_train, params)
    X_test_nn = x_scaler.transform(stock_test[features].values)
    X_test_nn_t = torch.tensor(X_test_nn, dtype=torch.float32)

    nn_model.eval()
    with torch.no_grad():
        nn_pred_scaled = nn_model(X_test_nn_t).numpy().flatten()

    nn_pred = y_scaler.inverse_transform(nn_pred_scaled.reshape(-1, 1)).flatten()

    out = stock_test[["date", "permno", target]].copy()
    out["ff3_pred"] = ff3_pred
    out["nn_pred"] = nn_pred
    return out


In [7]:
permnos = np.sort(pre["permno"].unique())

results = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(process_one_permno)(p, pre, test, BEST_PARAMS) for p in tqdm(permnos)
)

rows = [r for r in results if r is not None]
preds = pd.concat(rows, ignore_index=True)
preds.sort_values(["permno", "date"], inplace=True)

print("Predictions shape:", preds.shape)


100%|██████████| 2716/2716 [01:27<00:00, 31.14it/s]


Predictions shape: (684286, 5)


In [ ]:
mse_ff3 = np.mean((preds[target] - preds["ff3_pred"])**2)
mse_nn  = np.mean((preds[target] - preds["nn_pred"])**2)

print("STANDARD RETRAIN (used all pre-2024 data) — Test year 2024")
print("FF3 OOS MSE:", mse_ff3)
print("NN  OOS MSE:", mse_nn)
print("Delta (NN - FF3):", mse_nn - mse_ff3)

preds.to_parquet("Data/predictions_ff3_vs_nn_2024_standard_retrain.parquet", index=False)
print("Results Saved")



STANDARD RETRAIN (used all pre-2024 data) — Test year 2024
FF3 OOS MSE: 0.001920165841881584
NN  OOS MSE: 0.0019269391375083024
Delta (NN - FF3): 6.773295626718483e-06
Results Saved
